# Hybrid Retriever
> Fuse dense and sparse retrieval with Reciprocal Rank Fusion (RRF).

| Module | `anchor.retrieval` |
|--------|--------------------|
| Key classes | `HybridRetriever`, `DenseRetriever`, `SparseRetriever` |
| Difficulty | Intermediate |

`HybridRetriever` combines a dense retriever (semantic) with a sparse retriever
(keyword) and fuses their results using RRF weighted by configurable weights.

**Time:** ~7 minutes

## Setup

In [ ]:
from anchor.retrieval import DenseRetriever, SparseRetriever, HybridRetriever
from anchor.storage import InMemoryVectorStore, InMemoryContextStore
from anchor.models import ContextItem, SourceType, QueryBundle

## Build Dense and Sparse Retrievers

In [ ]:
# Shared stores and embedding function
vector_store = InMemoryVectorStore()
ctx_store = InMemoryContextStore()

def embed_fn(text: str) -> list[float]:
    padded = text[:128].ljust(128)
    return [hash(c) % 100 / 100.0 for c in padded]

dense = DenseRetriever(
    vector_store=vector_store,
    context_store=ctx_store,
    embed_fn=embed_fn,
)

sparse = SparseRetriever(context_store=ctx_store)

print("Dense and sparse retrievers created.")

## Index Shared Documents

In [ ]:
docs = [
    ("h-1", "Python is widely used for data science and machine learning."),
    ("h-2", "Rust ensures memory safety through its ownership model."),
    ("h-3", "Go is a statically typed language designed for concurrency."),
    ("h-4", "Machine learning requires large datasets and GPU compute."),
    ("h-5", "Data pipelines often use Python with Apache Spark."),
    ("h-6", "The ownership model in Rust prevents data races at compile time."),
]

items = [
    ContextItem(id=did, content=content, source=SourceType.RETRIEVAL,
               score=0.0, priority=5, token_count=15)
    for did, content in docs
]

dense.index(items)
sparse.index(items)
print(f"Indexed {len(items)} documents in both retrievers.")

## Create the Hybrid Retriever
Weights control the balance: `(0.7, 0.3)` means 70% dense, 30% sparse.

In [ ]:
hybrid = HybridRetriever(
    dense=dense,
    sparse=sparse,
    weights=(0.7, 0.3),
)

print(f"Hybrid weights: dense={0.7}, sparse={0.3}")

## Retrieve with RRF Fusion

In [ ]:
query = QueryBundle(query_str="machine learning data")
results = hybrid.retrieve(query, top_k=4)

print(f"Query: '{query.query_str}'")
print(f"Fused results ({len(results)}):\n")
for i, item in enumerate(results):
    print(f"  [{i+1}] score={item.score:.4f}  {item.content[:60]}")

## Experiment with Weights
Shift the balance toward sparse or dense.

In [ ]:
for w_dense, w_sparse in [(0.9, 0.1), (0.5, 0.5), (0.1, 0.9)]:
    h = HybridRetriever(dense=dense, sparse=sparse, weights=(w_dense, w_sparse))
    results = h.retrieve(query, top_k=3)
    top_id = results[0].id if results else "(none)"
    print(f"  weights=({w_dense}, {w_sparse}) -> top result: {top_id}")

## Key Takeaways
- `HybridRetriever` fuses dense (semantic) and sparse (keyword) results via RRF
- `weights=(dense, sparse)` controls the fusion balance
- Both sub-retrievers share the same indexed `ContextItem` objects
- Hybrid retrieval is more robust than either method alone
- Tune weights based on your query distribution

**Next:** [Scored Memory Retriever](04_scored_memory_retriever.ipynb)